# Preparation of the 2D Inputs of the Monocular Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nestpr/fit3d-3d-pose-benchmark/blob/main/notebooks/00_prepare_2d_inputs.ipynb)

This notebook builds the three sources of 2D keypoints that feed the monocular lifting models (TCPFormer and PoseMamba-L, notebooks 01 and 02). It is part of the code accompanying the paper *"Comprehensive Evaluation of Monocular and Multi-view 3D Human Pose Estimation Models for Fitness Exercise Analysis"* ([arXiv:2611.02929](https://arxiv.org/abs/2611.02929)).

| 2D source | How it is obtained | Section |
|---|---|---|
| `poses2d_gt` | Projection of the Fit3D 3D ground truth (`joints3d_25`) onto camera 65906101 with its calibration | 4 |
| `poses2d` (YOLO-Pose) | YOLO-Pose (Ultralytics) applied frame by frame to the video of camera 65906101 | 5 |
| `pose2d_rtmpose` (RTMPose) | **Provided as input data** (MMPose-style JSON files). Section 6 can optionally generate equivalent files with `rtmlib`. | 6 |

Section 7 validates the three sources (224 sequences each) and, optionally, compares them with a previous version of the files.

**Contents:** 1. Requirements · 2. Setup · 3. Configuration and data lookup · 4. Ground-truth 2D keypoints · 5. YOLO-Pose · 6. RTMPose · 7. Validation · 8. Summary

## 1. Requirements

**Runtime.** The experiments of the paper were run in **Google Colab**. Ground-truth projection and validation run on any computer (NumPy, OpenCV, Pandas). YOLO-Pose and RTMPose process every frame of 224 videos of 900 × 900 pixels, so a **GPU runtime** is recommended for them. Everything is stored under a single root folder, `STORAGE_ROOT`: Google Drive in Colab, the folder `data` next to the notebook otherwise.

**Data.** Download Fit3D from <https://fit3d.imar.ro> (academic license). The notebook expects:

```text
STORAGE_ROOT/
├── VIDEOS/<subject>/videos/<camera_id>/<exercise>.mp4      # RGB videos (only camera 65906101 is used here)
├── VIDEOS/<subject>/<exercise>/joints3d_25.json            # Fit3D 3D ground truth, in metres
├── camera_params/<subject>/<camera_id>/<exercise>.json     # calibration (intrinsics and extrinsics)
└── Fit3D_2D/<subject>/pose2d_rtmpose/<exercise>.json       # RTMPose detections (only if they are provided, see Section 6)
```

The same alternative locations as in notebook 03 are accepted (for example, `VIDEOS/<subject>/<camera_id>/<exercise>.mp4`, or the ground truth in `Fit3D_2D/<subject>/gt/<exercise>.json`).

**Output.** The notebook writes the 2D sources in the layout read by notebooks 01 and 02:

```text
STORAGE_ROOT/Fit3D_2D/<subject>/
├── poses2d_gt/<exercise>.npy      # (T, 17, 3): x, y, confidence = 1;  <exercise>.json: width, height, fps
├── poses2d/<exercise>.npy         # YOLO-Pose, (T, 17, 3);             <exercise>.json: width, height, fps
└── pose2d_rtmpose/<exercise>.json # RTMPose, MMPose-style JSON
```

**How to run.** Execute the cells from top to bottom. Every source can be switched on or off in Section 3. Sequences that already exist are skipped, so an interrupted run can be resumed.

## 2. Setup

In [ ]:
# Detect the runtime and mount Google Drive when needed

import importlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    import google.colab   # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Directory of the notebook (kept if this cell is run again)
PROJECT_DIR = globals().get("PROJECT_DIR") or os.getcwd()

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

print(f"Runtime: {'Google Colab' if IN_COLAB else 'local'} | Python {platform.python_version()}")

In [ ]:
# Imports

import cv2
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

print(f"OpenCV {cv2.__version__} | NumPy {np.__version__} | Pandas {pd.__version__}")


def ensure_package(import_name, pip_name=None):
    '''Import a package, installing it with pip the first time it is needed.'''
    try:
        return importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pip_name or import_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name], check=True)
        importlib.invalidate_caches()
        return importlib.import_module(import_name)

## 3. Configuration and Data Lookup

### 3.1 Settings

- `RUN_GT_2D`, `RUN_YOLO`, `RUN_RTMPOSE_GENERATION` select the sources that are generated. RTMPose is **off** by default because the detections used in the paper are provided as input data (Section 6).
- `YOLO_MODEL_NAME` is the Ultralytics checkpoint. It is downloaded automatically the first time it is used.
- `OUTPUT_2D_DIRNAME` is the folder of `STORAGE_ROOT` where the sources are written. Use a different name (for example `"Fit3D_2D_check"`) together with `REFERENCE_2D_DIRNAME = "Fit3D_2D"` to regenerate the files **without touching** the existing ones and compare both versions in Section 7 (the provided RTMPose files are not copied: they are read from, and generated ones written to, `OUTPUT_2D_DIRNAME`).
- `OVERWRITE = False` skips sequences that already have their output files.
- `MAX_SEQUENCES_PER_SUBJECT` and `MAX_FRAMES` limit the run to a few sequences and to their first frames, which is useful to test the notebook quickly. Leave both as `None` to generate the full inputs of the experiment.
- To try the notebook on a few sequences first, set `SUBJECTS` and `MAX_SEQUENCES_PER_SUBJECT`.

In [ ]:
# ----------------------------------------------------------------------------
# User-editable settings
# ----------------------------------------------------------------------------
STORAGE_ROOT = Path("/content/drive/MyDrive") if IN_COLAB else Path(PROJECT_DIR) / "data"

SUBJECTS = ["s03", "s04", "s05", "s07", "s08", "s09", "s10", "s11"]
CAMERA_ID = "65906101"                    # frontal camera used by all monocular experiments

RUN_GT_2D = True                          # Section 4: projection of the 3D ground truth
RUN_YOLO = True                           # Section 5: YOLO-Pose extraction (GPU recommended)
RUN_RTMPOSE_GENERATION = False            # Section 6: optional, the paper uses the files provided as input

YOLO_MODEL_NAME = "yolov8n-pose.pt"       # Ultralytics YOLO-Pose checkpoint
RTMPOSE_MODE = "performance"              # rtmlib preset: "performance", "balanced" or "lightweight"

OUTPUT_2D_DIRNAME = "Fit3D_2D"            # folder (inside STORAGE_ROOT) where the 2D sources are written
REFERENCE_2D_DIRNAME = None               # e.g. "Fit3D_2D" to compare a new OUTPUT_2D_DIRNAME with existing files
OVERWRITE = False                         # False: skip sequences whose outputs already exist

EXCLUDE_WARMUPS = True
MAX_SEQUENCES_PER_SUBJECT = None          # e.g. 2 for a quick test; None = all sequences
MAX_FRAMES = None                         # e.g. 250: only the first frames of every sequence (for tests); None = all
APPLY_LENS_DISTORTION = False             # project the ground truth with a pinhole model (see Section 4)
GT_INTRINSICS = "without_distortion"      # intrinsics block of the calibration file: "without_distortion", "with_distortion" or "auto"

DEFAULT_IMAGE_SIZE = {"width": 900, "height": 900, "fps": 50.0}   # Fit3D native video format

# Name of the folder of each source inside <OUTPUT_2D_ROOT>/<subject>/
SOURCE_FOLDERS = {"poses2d_gt": "poses2d_gt", "pose2d_yolo": "poses2d", "pose2d_rtmpose": "pose2d_rtmpose"}

# ----------------------------------------------------------------------------
# Derived locations
# ----------------------------------------------------------------------------
VIDEOS_ROOT = STORAGE_ROOT / "VIDEOS"
CAMERA_ROOT = STORAGE_ROOT / "camera_params"
OUTPUT_2D_ROOT = STORAGE_ROOT / OUTPUT_2D_DIRNAME
REFERENCE_2D_ROOT = STORAGE_ROOT / REFERENCE_2D_DIRNAME if REFERENCE_2D_DIRNAME else None

IS_SUBSET = MAX_SEQUENCES_PER_SUBJECT is not None or MAX_FRAMES is not None or len(SUBJECTS) != 8
EXPECTED_SEQUENCES = 224

print(f"STORAGE_ROOT: {STORAGE_ROOT}")
print(f"Output 2D sources: {OUTPUT_2D_ROOT}")
print(f"Reference for the comparison: {REFERENCE_2D_ROOT}")

### 3.2 Locating videos, calibration and ground truth

The list of sequences is built from the videos of camera `CAMERA_ID` (warm-ups excluded). If a subject has no videos, the exercises are taken from its ground-truth files instead, which is enough to project the ground truth.

In [ ]:
# Locate Fit3D videos, calibration and ground-truth files

import glob


def is_warmup(name):
    return str(name).lower().startswith("warmup")


def video_dirs(subject):
    '''Folders that may contain the videos of CAMERA_ID for a subject.'''
    return [
        VIDEOS_ROOT / subject / "videos" / CAMERA_ID,
        VIDEOS_ROOT / subject / CAMERA_ID,
    ]


def find_video_file(subject, exercise):
    '''Locate the video of CAMERA_ID for one sequence; return None when it does not exist.'''
    patterns = [
        VIDEOS_ROOT / subject / "videos" / CAMERA_ID / f"{exercise}.mp4",
        VIDEOS_ROOT / subject / CAMERA_ID / f"{exercise}.mp4",
        VIDEOS_ROOT / subject / "videos" / CAMERA_ID / f"*{exercise}*.mp4",
        VIDEOS_ROOT / subject / CAMERA_ID / f"*{exercise}*.mp4",
        VIDEOS_ROOT / f"**/{subject}/**/{CAMERA_ID}/**/{exercise}.mp4",
        VIDEOS_ROOT / f"**/{subject}/**/{CAMERA_ID}/**/*{exercise}*.mp4",
    ]

    for pattern in patterns:
        matches = sorted(glob.glob(str(pattern), recursive=True))

        if matches:
            return Path(matches[0])

    return None


def find_gt_json(subject, exercise):
    '''Locate the Fit3D 3D ground truth (joints3d_25) of one sequence; return None if it does not exist.'''
    candidates = [
        OUTPUT_2D_ROOT / subject / "gt" / f"{exercise}.json",
        STORAGE_ROOT / "Fit3D_2D" / subject / "gt" / f"{exercise}.json",
        STORAGE_ROOT / "Fit3D_2D" / subject / "gt (1)" / f"{exercise}.json",
        VIDEOS_ROOT / subject / exercise / "joints3d_25.json",
    ]

    for path in candidates:
        if path.is_file():
            return path

    return None


def find_camera_json(subject, exercise):
    '''Locate the calibration file of CAMERA_ID for one sequence; return None if it does not exist.'''
    base = CAMERA_ROOT / subject
    candidates = [
        base / CAMERA_ID / f"{exercise}.json",
        base / CAMERA_ID / f"{exercise} (1).json",
        base / "videos" / CAMERA_ID / f"{exercise}.json",
        base / "videos" / CAMERA_ID / f"{exercise} (1).json",
    ]

    for path in candidates:
        if path.is_file():
            return path

    if base.is_dir():
        scored = []

        for path in sorted(base.rglob("*.json")):
            text = str(path).lower()

            if CAMERA_ID.lower() in text:
                scored.append((10 + (10 if exercise.lower() in text else 0), path))

        if scored:
            return sorted(scored, key=lambda item: (-item[0], str(item[1])))[0][1]

    return None


def list_exercises(subject):
    '''Exercises of a subject: videos of CAMERA_ID if present, otherwise the ground-truth files.'''
    names = set()

    for folder in video_dirs(subject):
        if folder.is_dir():
            names.update(path.stem for path in folder.glob("*.mp4"))

    if not names:
        for folder in [VIDEOS_ROOT / subject, STORAGE_ROOT / "Fit3D_2D" / subject / "gt"]:
            if folder.is_dir():
                for path in folder.iterdir():
                    if path.is_dir() and (path / "joints3d_25.json").is_file():
                        names.add(path.name)
                    elif path.suffix == ".json" and folder.name == "gt":
                        names.add(path.stem)

    names = sorted(names)

    if EXCLUDE_WARMUPS:
        names = [name for name in names if not is_warmup(name)]

    if MAX_SEQUENCES_PER_SUBJECT is not None:
        names = names[:MAX_SEQUENCES_PER_SUBJECT]

    return names


def collect_sequences():
    return [(subject, exercise) for subject in SUBJECTS for exercise in list_exercises(subject)]


SEQUENCES = collect_sequences()
per_subject = pd.Series([s for s, _ in SEQUENCES]).value_counts().sort_index() if SEQUENCES else pd.Series(dtype=int)

print(f"Sequences found: {len(SEQUENCES)} from {len(per_subject)} subjects")
display(per_subject.rename("sequences").to_frame().T)

if not IS_SUBSET and len(SEQUENCES) != EXPECTED_SEQUENCES:
    print(f"WARNING: {EXPECTED_SEQUENCES} sequences (8 subjects x 28 exercises) are expected in the full experiment.")

In [ ]:
# Paths of the output files and video metadata

def source_dir(subject, source, root=None):
    return (root or OUTPUT_2D_ROOT) / subject / SOURCE_FOLDERS[source]


def output_paths(subject, exercise, source, root=None):
    '''(keypoint file, metadata file) of one sequence and source.'''
    folder = source_dir(subject, source, root)

    if source == "pose2d_rtmpose":
        return folder / f"{exercise}.json", None

    return folder / f"{exercise}.npy", folder / f"{exercise}.json"


def video_metadata(video_path):
    '''Width, height, fps and number of frames of a video; the defaults of Fit3D if there is no video.'''
    if video_path is None:
        return dict(DEFAULT_IMAGE_SIZE, frames=None)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Unable to open video: {video_path}")

    meta = {
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": float(cap.get(cv2.CAP_PROP_FPS)),
        "frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()

    return meta


def save_sequence(keypoints, npy_path, json_path, meta):
    npy_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(npy_path, np.asarray(keypoints, dtype=np.float32))

    with open(json_path, "w", encoding="utf-8") as file:
        json.dump({"width": meta["width"], "height": meta["height"], "fps": meta["fps"]}, file)


def is_done(subject, exercise, source):
    paths = [p for p in output_paths(subject, exercise, source) if p is not None]
    return all(p.is_file() for p in paths)

## 4. Ground-Truth 2D Keypoints

The ground-truth 2D source is the projection of the Fit3D 3D ground truth onto the image of camera `65906101`, which gives a reference free of detection errors. For every sequence:

1. The first 17 joints of `joints3d_25` (metres) are converted to millimetres.
2. They are transformed to the camera frame with the extrinsics of the calibration file, $X_{cam} = R\,(X - C)$, where $R$ is the rotation and $C$ the position of the camera in the world frame, which the Fit3D files store as `T` (notebook 03 converts it to the usual translation $-R\,T$). Units are millimetres: the camera position is converted from metres when needed.
3. They are projected with the pinhole model, $u = f_x\,X_{cam}/Z_{cam} + c_x$ and $v = f_y\,Y_{cam}/Z_{cam} + c_y$, using the block `intrinsics_wo_distortion` of the calibration file (`GT_INTRINSICS = "without_distortion"`). This is the projection that reproduces the ground-truth 2D files used in the paper (mean difference of 0.000 px on the test sequences). The calibration files also contain `intrinsics_w_distortion`, with a slightly different focal length and centre plus lens-distortion coefficients: `GT_INTRINSICS = "with_distortion"` selects them, `"auto"` uses them when present, and `APPLY_LENS_DISTORTION = True` applies the radial and tangential coefficients.

The joint order and the left/right labelling are those of `joints3d_25`: they are stored **unchanged**, and the exchange of the left and right legs that the lifting models need is applied when notebooks 01 and 02 load this source. A confidence of 1 is added to every keypoint, and the width, height and frame rate of the video are stored next to each file.

In [ ]:
# Projection of the 3D ground truth

def load_camera(subject, exercise, intrinsics_set=None):
    '''Extrinsics (rotation and camera position, in millimetres) and intrinsics of CAMERA_ID.'''
    path = find_camera_json(subject, exercise)

    if path is None:
        raise FileNotFoundError(f"Calibration not found for {subject}/{CAMERA_ID}/{exercise} under {CAMERA_ROOT}")

    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    rotation = np.asarray(data["extrinsics"]["R"], dtype=np.float64).reshape(3, 3)
    position = np.asarray(data["extrinsics"]["T"], dtype=np.float64).reshape(3)   # camera position in the world frame

    if np.nanmax(np.abs(position)) < 100:   # stored in metres (same rule as notebook 03)
        position = position * 1000.0

    choice = intrinsics_set or GT_INTRINSICS
    key = {"auto": "intrinsics_w_distortion" if "intrinsics_w_distortion" in data else "intrinsics_wo_distortion",
           "with_distortion": "intrinsics_w_distortion",
           "without_distortion": "intrinsics_wo_distortion"}[choice]

    if key not in data:
        raise KeyError(f"Missing {key} in {path}")

    intrinsics = data[key]
    fx, fy = np.asarray(intrinsics["f"], dtype=np.float64).reshape(-1)[:2]
    cx, cy = np.asarray(intrinsics["c"], dtype=np.float64).reshape(-1)[:2]
    k = np.zeros(3)
    p = np.zeros(2)

    if key == "intrinsics_w_distortion":
        k[:] = np.asarray(intrinsics.get("k", [0.0, 0.0, 0.0]), dtype=np.float64).reshape(-1)[:3]
        p[:] = np.asarray(intrinsics.get("p", [0.0, 0.0]), dtype=np.float64).reshape(-1)[:2]

    return {"R": rotation, "C": position, "fx": fx, "fy": fy, "cx": cx, "cy": cy, "k": k, "p": p}


def project_points(points_mm, camera, lens_distortion=False):
    '''Project world points (N, 3) in millimetres to pixels (N, 2).'''
    points_cam = (points_mm - camera["C"]) @ camera["R"].T
    z = points_cam[:, 2:3]
    z = np.where(np.abs(z) < 1e-6, 1e-6, z)

    if not lens_distortion:
        u = camera["fx"] * points_cam[:, 0:1] / z + camera["cx"]
        v = camera["fy"] * points_cam[:, 1:2] / z + camera["cy"]
        return np.concatenate([u, v], axis=1)

    matrix = np.array([[camera["fx"], 0.0, camera["cx"]], [0.0, camera["fy"], camera["cy"]], [0.0, 0.0, 1.0]])
    distortion = np.array([camera["k"][0], camera["k"][1], camera["p"][0], camera["p"][1], camera["k"][2]])
    pixels, _ = cv2.projectPoints(points_cam.reshape(-1, 1, 3), np.zeros(3), np.zeros(3), matrix, distortion)

    return pixels.reshape(-1, 2)


def project_ground_truth(subject, exercise, lens_distortion=None, intrinsics_set=None):
    '''Ground-truth 2D keypoints (T, 17, 3) of one sequence: x, y in pixels and confidence 1.'''
    gt_path = find_gt_json(subject, exercise)

    if gt_path is None:
        raise FileNotFoundError(f"Ground truth (joints3d_25) not found for {subject}/{exercise}")

    with open(gt_path, "r", encoding="utf-8") as file:
        gt = np.asarray(json.load(file)["joints3d_25"], dtype=np.float64)

    if gt.ndim == 4 and gt.shape[1] == 1:
        gt = gt[:, 0]

    if gt.ndim != 3 or gt.shape[1:] != (25, 3):
        raise ValueError(f"Unexpected ground-truth shape {gt.shape} in {gt_path}")

    if MAX_FRAMES is not None:
        gt = gt[:MAX_FRAMES]

    gt_mm = gt[:, :17, :] * 1000.0
    camera = load_camera(subject, exercise, intrinsics_set)
    use_distortion = APPLY_LENS_DISTORTION if lens_distortion is None else lens_distortion
    pixels = project_points(gt_mm.reshape(-1, 3), camera, use_distortion).reshape(len(gt), 17, 2)

    return np.concatenate([pixels, np.ones((len(gt), 17, 1))], axis=-1).astype(np.float32)

In [ ]:
# Generate the ground-truth 2D source

def run_ground_truth_2d():
    failed = []
    done = skipped = 0

    for subject, exercise in tqdm(SEQUENCES, desc="Ground-truth 2D"):
        npy_path, json_path = output_paths(subject, exercise, "poses2d_gt")

        if is_done(subject, exercise, "poses2d_gt") and not OVERWRITE:
            skipped += 1
            continue

        try:
            keypoints = project_ground_truth(subject, exercise)
            meta = video_metadata(find_video_file(subject, exercise))
            save_sequence(keypoints, npy_path, json_path, meta)
            done += 1
        except Exception as error:
            failed.append((subject, exercise, str(error)))
            print(f"ERROR | {subject}/{exercise} | {error}")

    print(f"Ground-truth 2D | generated: {done} | skipped (already existed): {skipped} | failed: {len(failed)}")

    return failed


if RUN_GT_2D:
    failed_gt = run_ground_truth_2d()
else:
    print("RUN_GT_2D = False: ground-truth 2D keypoints are not generated.")

## 5. YOLO-Pose

YOLO-Pose (Maji et al., CVPRW 2022) is a single-stage detector that predicts the person and its 17 COCO keypoints in one pass. It is run with the [Ultralytics](https://github.com/ultralytics/ultralytics) implementation and the checkpoint `YOLO_MODEL_NAME`, which is downloaded automatically. For every frame of the video of camera `65906101`:

- The detector is run with the default Ultralytics settings (no confidence threshold or image size is changed) and the **first detected person** is kept. The detections are sorted by confidence, so this is the most confident one. Each keypoint is stored as $(x, y, \text{confidence})$ in pixels.
- If no person is detected, the keypoints of the previous frame are repeated (zeros if it happens in the first frame).
- The sequence of COCO-17 keypoints is converted to the 17-joint Human3.6M layout of the lifting models: Pelvis is the midpoint of the hips, Neck the midpoint of the shoulders, Spine the midpoint of Pelvis and Neck, and Nose and Head both take the COCO nose keypoint (there is no head-top keypoint). The COMMON15 skeleton of the paper uses Nose and discards Spine and Head.

The file `<exercise>.json` stores the width, height and frame rate of the video, which are needed to normalize the coordinates in notebooks 01 and 02.

In [ ]:
# COCO-17 -> Human3.6M-17 conversion and YOLO-Pose extraction

def coco17_to_h36m17(kpts):
    '''Convert COCO-17 keypoints (T, 17, 3) to the 17-joint Human3.6M layout of the lifting models.'''
    NOSE = 0
    L_SHO, R_SHO = 5, 6
    L_ELB, R_ELB = 7, 8
    L_WRI, R_WRI = 9, 10
    L_HIP, R_HIP = 11, 12
    L_KNE, R_KNE = 13, 14
    L_ANK, R_ANK = 15, 16

    out = np.zeros((kpts.shape[0], 17, 3), dtype=np.float32)

    pelvis = (kpts[:, L_HIP] + kpts[:, R_HIP]) / 2.0
    neck = (kpts[:, L_SHO] + kpts[:, R_SHO]) / 2.0
    spine = (pelvis + neck) / 2.0
    head = kpts[:, NOSE].copy()

    out[:, 0] = pelvis
    out[:, 1] = kpts[:, R_HIP]
    out[:, 2] = kpts[:, R_KNE]
    out[:, 3] = kpts[:, R_ANK]
    out[:, 4] = kpts[:, L_HIP]
    out[:, 5] = kpts[:, L_KNE]
    out[:, 6] = kpts[:, L_ANK]
    out[:, 7] = spine
    out[:, 8] = neck
    out[:, 9] = head
    out[:, 10] = head
    out[:, 11] = kpts[:, L_SHO]
    out[:, 12] = kpts[:, L_ELB]
    out[:, 13] = kpts[:, L_WRI]
    out[:, 14] = kpts[:, R_SHO]
    out[:, 15] = kpts[:, R_ELB]
    out[:, 16] = kpts[:, R_WRI]

    return out


def extract_yolo_sequence(pose_model, video_path):
    '''Run YOLO-Pose frame by frame and return the (T, 17, 3) Human3.6M-layout keypoints.'''
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Unable to open video: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total = min(total, MAX_FRAMES) if MAX_FRAMES is not None else total
    frames = []
    previous = None

    with tqdm(total=total, desc=Path(video_path).stem, leave=False) as progress:
        while True:
            ok, frame = cap.read()

            if not ok or (MAX_FRAMES is not None and len(frames) >= MAX_FRAMES):
                break

            result = pose_model(frame, verbose=False)[0]

            if len(result.keypoints) == 0:
                frames.append(previous if previous is not None else np.zeros((17, 3), dtype=np.float32))
            else:
                xy = result.keypoints.xy[0].cpu().numpy()
                conf = result.keypoints.conf[0].cpu().numpy()
                current = np.concatenate([xy, conf[:, None]], axis=1).astype(np.float32)
                frames.append(current)
                previous = current

            progress.update(1)

    cap.release()

    if not frames:
        raise RuntimeError(f"No frames could be read from {video_path}")

    return coco17_to_h36m17(np.stack(frames).astype(np.float32))

In [ ]:
# Generate the YOLO-Pose source

def run_yolo():
    ensure_package("ultralytics")
    from ultralytics import YOLO

    pose_model = YOLO(YOLO_MODEL_NAME)
    print(f"YOLO-Pose checkpoint: {YOLO_MODEL_NAME}")

    failed = []
    done = skipped = 0

    for index, (subject, exercise) in enumerate(SEQUENCES, start=1):
        npy_path, json_path = output_paths(subject, exercise, "pose2d_yolo")

        if is_done(subject, exercise, "pose2d_yolo") and not OVERWRITE:
            skipped += 1
            continue

        video_path = find_video_file(subject, exercise)

        if video_path is None:
            failed.append((subject, exercise, "video not found"))
            print(f"SKIP | {subject}/{exercise} | video of camera {CAMERA_ID} not found")
            continue

        start = time.time()

        try:
            keypoints = extract_yolo_sequence(pose_model, video_path)
            save_sequence(keypoints, npy_path, json_path, video_metadata(video_path))
            done += 1
            print(f"[{index}/{len(SEQUENCES)}] {subject}/{exercise} | {keypoints.shape} | {time.time() - start:.0f} s")
        except Exception as error:
            failed.append((subject, exercise, str(error)))
            print(f"ERROR | {subject}/{exercise} | {error}")

    print(f"YOLO-Pose | generated: {done} | skipped (already existed): {skipped} | failed: {len(failed)}")

    return failed


if RUN_YOLO:
    failed_yolo = run_yolo()
else:
    print("RUN_YOLO = False: YOLO-Pose keypoints are not generated.")

## 6. RTMPose

[RTMPose](https://arxiv.org/abs/2303.07399) is a top-down detector (a person detector followed by a keypoint model) that outputs COCO-17 keypoints. **The RTMPose detections used in the paper were produced outside this repository and are provided as input data**: one MMPose-style JSON file per sequence in `Fit3D_2D/<subject>/pose2d_rtmpose/<exercise>.json`. Notebooks 01 and 02 read them directly.

### Format

Each file is a list with one entry per frame:

```json
[
  {"frame_id": 0,
   "instances": [{"keypoints": [[x, y], "... 17 pairs"],
                  "keypoint_scores": [0.97, "... 17 values"],
                  "bbox": [[x1, y1, x2, y2]],
                  "bbox_score": 0.99}]},
  "..."
]
```

When a frame has several instances, the one with the highest `bbox_score` is used. A frame without any instance repeats the most recent valid pose (and the frames before the first detection use the first valid one). Section 7 validates the files against this format.

### Optional: generating equivalent files

To run the pipeline on other data, or without the provided files, `RUN_RTMPOSE_GENERATION = True` produces files in the same format with [`rtmlib`](https://github.com/Tau-J/rtmlib), a lightweight package that runs the RTMPose models with ONNX Runtime (no MMPose or MMCV installation is needed).

> **These files are not identical to the ones used in the paper.** The model variant, detector and thresholds depend on the tool that produced the original detections, and `rtmlib` returns neither a detection score nor the detection box, so the mean keypoint score of the instance is stored as `bbox_score` and the extent of its keypoints as `bbox`. The PA-MPJPE obtained with generated files can therefore differ slightly from the RTMPose rows of the paper; the values of the paper are reproduced with the provided files. Existing files are never overwritten unless `OVERWRITE = True`.

In [ ]:
# Optional generation of RTMPose detections with rtmlib

def make_rtmpose_frame(frame_id, keypoints, scores):
    '''One entry of the MMPose-style JSON list from the output of rtmlib: keypoints (N, 17, 2), scores (N, 17).'''
    keypoints = np.asarray(keypoints, dtype=np.float64).reshape(-1, 17, 2)
    scores = np.asarray(scores, dtype=np.float64).reshape(-1, 17)
    instances = []

    for points, point_scores in zip(keypoints, scores):
        box = [points[:, 0].min(), points[:, 1].min(), points[:, 0].max(), points[:, 1].max()]   # extent of the keypoints
        instances.append({
            "keypoints": points.tolist(),
            "keypoint_scores": point_scores.tolist(),
            "bbox": [[float(value) for value in box]],
            "bbox_score": float(point_scores.mean()),   # rtmlib does not return the detection score
        })

    return {"frame_id": int(frame_id), "instances": instances}


def run_rtmpose_generation():
    ensure_package("onnxruntime", "onnxruntime-gpu" if shutil.which("nvidia-smi") else "onnxruntime")
    rtmlib = ensure_package("rtmlib")

    import onnxruntime

    device = "cuda" if "CUDAExecutionProvider" in onnxruntime.get_available_providers() else "cpu"

    if device == "cpu":
        print("WARNING: ONNX Runtime has no GPU provider, so RTMPose runs on the CPU and is slow "
              "(on a GPU runtime install 'onnxruntime-gpu' instead of 'onnxruntime').")

    body = rtmlib.Body(mode=RTMPOSE_MODE, backend="onnxruntime", device=device, to_openpose=False)
    print(f"rtmlib Body | mode: {RTMPOSE_MODE} | device: {device}")

    failed = []
    done = skipped = 0

    for index, (subject, exercise) in enumerate(SEQUENCES, start=1):
        json_path, _ = output_paths(subject, exercise, "pose2d_rtmpose")

        if json_path.is_file() and not OVERWRITE:
            skipped += 1
            continue

        video_path = find_video_file(subject, exercise)

        if video_path is None:
            failed.append((subject, exercise, "video not found"))
            print(f"SKIP | {subject}/{exercise} | video of camera {CAMERA_ID} not found")
            continue

        start = time.time()

        try:
            cap = cv2.VideoCapture(str(video_path))
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            total = min(total, MAX_FRAMES) if MAX_FRAMES is not None else total
            frames = []

            with tqdm(total=total, desc=exercise, leave=False) as progress:
                frame_id = 0

                while True:
                    ok, image = cap.read()

                    if not ok or (MAX_FRAMES is not None and frame_id >= MAX_FRAMES):
                        break

                    keypoints, scores = body(image)
                    frames.append(make_rtmpose_frame(frame_id, keypoints, scores))
                    frame_id += 1
                    progress.update(1)

            cap.release()

            json_path.parent.mkdir(parents=True, exist_ok=True)

            with open(json_path, "w", encoding="utf-8") as file:
                json.dump(frames, file)

            done += 1
            print(f"[{index}/{len(SEQUENCES)}] {subject}/{exercise} | {len(frames)} frames | {time.time() - start:.0f} s")
        except Exception as error:
            failed.append((subject, exercise, str(error)))
            print(f"ERROR | {subject}/{exercise} | {error}")

    print(f"RTMPose | generated: {done} | skipped (already existed): {skipped} | failed: {len(failed)}")

    return failed


if RUN_RTMPOSE_GENERATION:
    failed_rtmpose = run_rtmpose_generation()
else:
    print("RUN_RTMPOSE_GENERATION = False: the provided RTMPose files are used as they are (validated in Section 7).")

## 7. Validation

Every source is checked for the 224 sequences of the experiment (8 subjects × 28 exercises, warm-ups excluded):

| Check | Applies to | Result if it fails |
|---|---|---|
| The file exists, and so does its `.json` metadata (`width`, `height`) | all | `MISSING` / `FAIL` |
| Shape `(T, 17, 3)`, finite values | `poses2d_gt`, `poses2d` | `FAIL` |
| At least 95 % of the keypoints inside the image (a wrong calibration or unit puts them outside) | `poses2d_gt`, `poses2d` | `WARN` / `FAIL` below 50 % |
| MMPose-style structure: list of frames, instances with 17 `keypoints`, `keypoint_scores` and a `bbox_score` | `pose2d_rtmpose` | `FAIL` |
| Fraction of frames with at least one detection | `pose2d_rtmpose` | `WARN` below 99 % |
| Number of frames equal to the ground truth | all | `WARN` (notebooks 01 and 02 use the shorter of the two) |

### Comparison with an existing version

When `REFERENCE_2D_DIRNAME` is set (Section 3.1), each sequence of `OUTPUT_2D_DIRNAME` is compared with the same file of the reference folder: mean distance in pixels between the keypoints. This is the way to confirm that the notebook reproduces files that were generated earlier: a different YOLO checkpoint, for instance, gives differences of several pixels instead of a fraction of a pixel.

In [ ]:
# Validation of the 2D sources

from functools import lru_cache


@lru_cache(maxsize=None)
def ground_truth_frames(subject, exercise):
    path = find_gt_json(subject, exercise)

    if path is None:
        return None

    with open(path, "r", encoding="utf-8") as file:
        frames = len(json.load(file)["joints3d_25"])

    return min(frames, MAX_FRAMES) if MAX_FRAMES is not None else frames


def read_metadata(json_path):
    '''Width, height of a metadata file; None if it is missing or incomplete.'''
    if json_path is None or not json_path.is_file():
        return None

    try:
        with open(json_path, "r", encoding="utf-8") as file:
            meta = json.load(file)
    except Exception:
        return None

    return meta if isinstance(meta, dict) and "width" in meta and "height" in meta else None


def load_rtmpose_xy(path):
    '''Read an MMPose-style file; return (T, 17, 2), the fraction of frames with detections and the frame ids.'''
    with open(path, "r", encoding="utf-8") as file:
        frames = json.load(file)

    if not isinstance(frames, list) or len(frames) == 0:
        raise ValueError("the file must contain a non-empty list of frames")

    ids = [int(frame.get("frame_id", index)) for index, frame in enumerate(frames)]
    num_frames = max(ids) + 1
    xy = np.full((num_frames, 17, 2), np.nan, dtype=np.float32)

    for frame_id, frame in zip(ids, frames):
        instances = frame.get("instances", [])

        if len(instances) == 0:
            continue

        best = max(instances, key=lambda instance: float(instance.get("bbox_score", 0.0)))
        points = np.asarray(best["keypoints"], dtype=np.float32)

        if points.ndim != 2 or points.shape[0] < 17:
            raise ValueError(f"frame {frame_id}: fewer than 17 keypoints")

        if len(best.get("keypoint_scores", [])) < 17:
            raise ValueError(f"frame {frame_id}: fewer than 17 keypoint scores")

        xy[frame_id] = points[:17, :2]

    valid = ~np.isnan(xy).any(axis=(1, 2))

    if not valid.any():
        raise ValueError("no frame contains a detection")

    # Same rule as notebooks 01 and 02: repeat the most recent valid pose
    last_valid = np.maximum.accumulate(np.where(valid, np.arange(num_frames), -1))
    last_valid[last_valid < 0] = np.flatnonzero(valid)[0]

    return xy[last_valid], float(valid.mean()), ids


def load_keypoints_xy(subject, exercise, source, root=None):
    '''Keypoints (T, J, 2) of a stored sequence, or None if the file does not exist.'''
    keypoint_path, _ = output_paths(subject, exercise, source, root)

    if not keypoint_path.is_file():
        return None

    if source == "pose2d_rtmpose":
        return load_rtmpose_xy(keypoint_path)[0]

    return np.load(keypoint_path)[..., :2]


def validate_sequence(subject, exercise, source):
    row = {"source": source, "subject": subject, "exercise": exercise, "status": "OK", "frames": None,
           "gt_frames": ground_truth_frames(subject, exercise), "notes": []}
    keypoint_path, json_path = output_paths(subject, exercise, source)

    if not keypoint_path.is_file():
        row.update(status="MISSING")
        return row

    try:
        if source == "pose2d_rtmpose":
            xy, detection_rate, ids = load_rtmpose_xy(keypoint_path)
            row["frames"] = len(xy)

            if sorted(ids) != list(range(min(ids), min(ids) + len(ids))):
                row["notes"].append("frame_id values are not consecutive")
            elif min(ids) != 0:
                row["notes"].append(f"frame_id starts at {min(ids)} (0 expected)")

            if detection_rate < 0.99:
                row["notes"].append(f"detections in {100 * detection_rate:.1f} % of the frames")
                row["status"] = "WARN"
        else:
            array = np.load(keypoint_path)

            if array.ndim != 3 or array.shape[1:] != (17, 3):
                raise ValueError(f"unexpected shape {array.shape}")

            if not np.isfinite(array).all():
                raise ValueError("NaN or Inf values")

            row["frames"] = len(array)
            meta = read_metadata(json_path)

            if meta is None:
                raise ValueError("metadata file (width, height) missing or incomplete")

            inside = ((array[..., 0] >= 0) & (array[..., 0] <= meta["width"])
                      & (array[..., 1] >= 0) & (array[..., 1] <= meta["height"])).mean()

            if inside < 0.5:
                raise ValueError(f"only {100 * inside:.0f} % of the keypoints are inside the image")

            if inside < 0.95:
                row["notes"].append(f"{100 * inside:.1f} % of the keypoints inside the image")
                row["status"] = "WARN"
    except Exception as error:
        row.update(status="FAIL")
        row["notes"].append(str(error))

    if row["status"] != "FAIL" and row["gt_frames"] is not None and row["frames"] != row["gt_frames"]:
        row["notes"].append(f"frames: {row['frames']} vs ground truth {row['gt_frames']}")

        if row["status"] == "OK":
            row["status"] = "WARN"

    row["notes"] = "; ".join(row["notes"])

    return row


def active_sources():
    '''Sources to validate: the ones generated in this run or already present in the output folder.'''
    flags = {"poses2d_gt": RUN_GT_2D, "pose2d_yolo": RUN_YOLO, "pose2d_rtmpose": RUN_RTMPOSE_GENERATION}
    active = [source for source in SOURCE_FOLDERS
              if flags[source] or any(source_dir(subject, source).is_dir() for subject in SUBJECTS)]

    for source in SOURCE_FOLDERS:
        if source not in active:
            print(f"Not validated (no files in {OUTPUT_2D_ROOT} and not generated): {source}")

    return active


def validate_all():
    rows = []

    for source in active_sources():
        for subject, exercise in tqdm(SEQUENCES, desc=f"Validating {source}", leave=False):
            rows.append(validate_sequence(subject, exercise, source))

    return pd.DataFrame(rows)


validation = validate_all()

summary_counts = (validation.pivot_table(index="source", columns="status", values="exercise", aggfunc="count", fill_value=0)
                  .reindex(columns=["OK", "WARN", "FAIL", "MISSING"], fill_value=0))
summary_counts.insert(0, "expected", len(SEQUENCES))

display(summary_counts)

problems = validation[validation["status"].isin(["WARN", "FAIL"])]

if len(problems):
    print(f"\n{len(problems)} sequences with warnings or failures (first 20):")
    display(problems.head(20))

In [ ]:
# Comparison with a previous version of the files (optional)

REFERENCE_TOLERANCE_PX = {"poses2d_gt": 0.5, "pose2d_yolo": 2.0, "pose2d_rtmpose": 2.0}   # mean error per sequence (pixels)
REFERENCE_HINT = {
    "poses2d_gt": "check APPLY_LENS_DISTORTION, GT_INTRINSICS and the calibration files",
    "pose2d_yolo": "check YOLO_MODEL_NAME (a different checkpoint gives differences of several pixels)",
    "pose2d_rtmpose": "generated files are not expected to match the provided ones exactly",
}


def compare_with_reference():
    if REFERENCE_2D_ROOT is None or not REFERENCE_2D_ROOT.is_dir():
        print("No reference folder: set REFERENCE_2D_DIRNAME in Section 3.1 to compare with existing files.")
        return pd.DataFrame()

    if REFERENCE_2D_ROOT.resolve() == OUTPUT_2D_ROOT.resolve():
        print("The reference folder is the output folder; use a different OUTPUT_2D_DIRNAME to compare two versions.")
        return pd.DataFrame()

    rows = []

    for source in active_sources():
        for subject, exercise in SEQUENCES:
            try:
                new = load_keypoints_xy(subject, exercise, source)
                old = load_keypoints_xy(subject, exercise, source, REFERENCE_2D_ROOT)
            except Exception as error:
                rows.append({"source": source, "subject": subject, "exercise": exercise, "mean_error_px": np.nan, "note": str(error)})
                continue

            if new is None or old is None:
                continue

            n = min(len(new), len(old))
            error = np.linalg.norm(new[:n] - old[:n], axis=-1).mean()
            rows.append({"source": source, "subject": subject, "exercise": exercise, "mean_error_px": float(error),
                         "frames_new": len(new), "frames_reference": len(old)})

    comparison = pd.DataFrame(rows)

    if comparison.empty:
        print("No sequence is present in both folders.")
        return comparison

    report = []

    for source, group in comparison.groupby("source"):
        errors = group["mean_error_px"].dropna()
        tolerance = REFERENCE_TOLERANCE_PX[source]
        share = float((errors < tolerance).mean()) if len(errors) else 0.0
        same_frames = float((group["frames_new"] == group["frames_reference"]).mean())
        report.append({
            "source": source, "sequences": len(group),
            "median error (px)": errors.median(), "95th percentile (px)": errors.quantile(0.95), "max (px)": errors.max(),
            f"share below tolerance": share, "same frame count": same_frames,
            "result": "MATCHES" if share >= 0.98 and same_frames >= 0.98 else "DIFFERS",
        })

    report = pd.DataFrame(report)
    display(report.round(3))

    for row in report.itertuples():
        if row.result == "DIFFERS":
            print(f"{row.source}: differs from the reference ({REFERENCE_HINT[row.source]}).")

    return comparison


reference_comparison = compare_with_reference()

## 8. Summary and Next Steps

**Files generated** (under `OUTPUT_2D_ROOT`, see the layout in Section 1) are the inputs of notebooks 01 and 02. Run them once per source, setting `POSE_SOURCE` to `"poses2d_gt"`, `"pose2d_yolo"` or `"pose2d_rtmpose"`. With the full experiment (224 sequences) the PA-MPJPE of the 17-joint skeleton should be:

| 2D source | `POSE_SOURCE` | TCPFormer | PoseMamba-L |
|---|---|---:|---:|
| Ground truth | `poses2d_gt` | 46.16 | 37.12 |
| YOLO-Pose | `pose2d_yolo` | 93.61 | 84.32 |
| RTMPose | `pose2d_rtmpose` | 70.89 | 71.14 |

Notebook 05 (Table 4) checks these values against the paper.

### Troubleshooting

| Message or symptom | Cause and solution |
|---|---|
| `Sequences found: 0` | The videos or ground-truth files are not where Section 3.2 looks. Check `STORAGE_ROOT` and the layout in Section 1. |
| `Calibration not found` | `camera_params/<subject>/65906101/<exercise>.json` is missing (or the camera id differs). |
| `poses2d_gt` validation shows keypoints outside the image | The ground truth or the calibration is in unexpected units, or the wrong camera is used. |
| YOLO-Pose differs from the reference by several pixels | A different checkpoint was used; set `YOLO_MODEL_NAME` to the one that produced the reference files. |
| YOLO-Pose is slow | Switch the runtime to GPU (*Runtime → Change runtime type*). Each sequence takes the time of running the detector on every frame. |
| `WARN` frame count differs from the ground truth | The video and the motion capture differ by some frames; notebooks 01 and 02 evaluate on the shorter of the two. |